# 🚀 Approach 3: Two-Stage 10-Fold Stacking Meta-Learner (Linear Signals + Tree Splits)

## 🧠 The Core Hypothesis:
The ultimate Kaggle benchmark strategy blends two completely different algorithmic families:
1. **Level-0 Diverse Base Models (10-Fold Out-of-Fold, 90% Train Data per Fold):**
   - Model 1: `LogisticRegression(L2)` (Global Linear Hyperplane)
   - Model 2: `LogisticRegression(L1/Lasso)` (Sparse Feature Selection)
   - Model 3: `SGDClassifier(ElasticNet)` (Asymmetric Loss Optimization)
   - Model 4: `HistGradientBoostingClassifier` (Shallow Trees `depth=4` for Non-Linear Interactions)
   - Model 5: `ExtraTreesClassifier` (Extremely Randomized Trees for Variance Reduction)
2. **Level-1 Meta-Learner:** A regularized `LogisticRegression` is trained directly on the $N \times 5$ OOF probability matrix. It automatically finds optimal non-linear blending weights.
3. **High-Resolution Threshold Scan (step=0.001):** Pin-points the exact F1 maximum.

**Hardware:** 100% CPU Friendly (Runs in ~3-4 minutes).


In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
N_FOLDS = 10
print('=' * 75)
print('  APPROACH 3: TWO-STAGE 10-FOLD STACKING META-LEARNER')
print('=' * 75)


In [ ]:
CANDIDATE_DIRS = [
    '/kaggle/input/competitions/pstu-data-thon-2026-vol-1',
    '/kaggle/input/pstu-data-thon-2026-vol-1',
    'pstu-data-thon-2026-vol-1',
    '../input/competitions/pstu-data-thon-2026-vol-1',
    './Dataset',
    '.'
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(os.path.join(d, 'train.csv'))), None)
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

TARGET_COL = 'TARGET'
y = train_raw[TARGET_COL].copy()
test_ids = test_raw['id'].copy() if 'id' in test_raw.columns else pd.Series(range(len(test_raw)), name='id')
X_tr_raw = train_raw.drop(columns=[TARGET_COL])
X_te_raw = test_raw.drop(columns=['id']) if 'id' in test_raw.columns else test_raw.copy()

feat_cols = [c for c in X_tr_raw.columns if c.startswith('feat_')]
cat_cols  = X_tr_raw[feat_cols].select_dtypes(include=['object']).columns.tolist()
num_cols  = [c for c in feat_cols if c not in cat_cols]

X_num_tr = X_tr_raw[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)
X_num_te = X_te_raw[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

# Outlier Clipping
p_low = np.percentile(X_num_tr, 1, axis=0)
p_high = np.percentile(X_num_tr, 99, axis=0)
X_num_tr = pd.DataFrame(np.clip(X_num_tr.values, p_low, p_high), columns=num_cols)
X_num_te = pd.DataFrame(np.clip(X_num_te.values, p_low, p_high), columns=num_cols)

scaler = RobustScaler()
X_tr_scaled = np.nan_to_num(scaler.fit_transform(X_num_tr), nan=0.0).astype(np.float32)
X_te_scaled = np.nan_to_num(scaler.transform(X_num_te), nan=0.0).astype(np.float32)
print(f'Features preprocessed: {X_tr_scaled.shape}')


## 2. Train Level-0 Base Models (10-Fold CV) & Construct Out-Of-Fold Meta-Matrix

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

base_models = {
    'L2_Logistic': lambda: LogisticRegression(C=0.05, penalty='l2', class_weight='balanced', max_iter=500, random_state=SEED),
    'L1_Lasso': lambda: LogisticRegression(C=0.02, penalty='l1', solver='saga', class_weight='balanced', max_iter=300, random_state=SEED),
    'SGD_Elastic': lambda: SGDClassifier(loss='log_loss', penalty='elasticnet', alpha=1e-3, class_weight='balanced', max_iter=500, random_state=SEED),
    'HistGBDT': lambda: HistGradientBoostingClassifier(max_depth=4, l2_regularization=5.0, class_weight='balanced', max_iter=150, random_state=SEED),
    'ExtraTrees': lambda: ExtraTreesClassifier(n_estimators=100, max_depth=8, min_samples_leaf=30, class_weight='balanced', n_jobs=-1, random_state=SEED)
}

oof_meta = np.zeros((len(y), len(base_models)), dtype=np.float32)
test_meta = np.zeros((len(X_te_scaled), len(base_models)), dtype=np.float32)

for m_idx, (m_name, m_fn) in enumerate(base_models.items()):
    t_start = time.time()
    print(f'Training Level-0 Model {m_idx+1}/{len(base_models)}: {m_name} across {N_FOLDS} Folds...')
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_scaled, y)):
        X_tr, y_tr = X_tr_scaled[tr_idx], y.iloc[tr_idx].values
        X_va, y_va = X_tr_scaled[va_idx], y.iloc[va_idx].values
        
        clf = m_fn()
        clf.fit(X_tr, y_tr)
        
        oof_meta[va_idx, m_idx] = clf.predict_proba(X_va)[:, 1]
        test_meta[:, m_idx] += clf.predict_proba(X_te_scaled)[:, 1] / N_FOLDS
        
    auc = roc_auc_score(y, oof_meta[:, m_idx])
    print(f'  {m_name} 10-Fold OOF AUC: {auc:.5f} [{time.time() - t_start:.1f}s]')


## 3. Train Level-1 Stacking Meta-Learner (10-Fold CV)

In [ ]:
# Level-1 Meta-Learner: Learns optimal blending weights from OOF probability matrix
oof_stacked = np.zeros(len(y), dtype=np.float32)
test_stacked = np.zeros(len(X_te_scaled), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(oof_meta, y)):
    meta_tr, y_tr = oof_meta[tr_idx], y.iloc[tr_idx].values
    meta_va, y_va = oof_meta[va_idx], y.iloc[va_idx].values
    
    meta_learner = LogisticRegression(C=1.0, max_iter=300, random_state=SEED)
    meta_learner.fit(meta_tr, y_tr)
    
    oof_stacked[va_idx] = meta_learner.predict_proba(meta_va)[:, 1]
    test_stacked += meta_learner.predict_proba(test_meta)[:, 1] / N_FOLDS

# Optimize Threshold
thresholds = np.arange(0.05, 0.95, 0.001)
best_f1, best_t = 0.0, 0.5
for t in thresholds:
    b = (oof_stacked >= t).astype(int)
    if b.sum() == 0: continue
    f = f1_score(y.values, b)
    if f > best_f1: best_f1, best_t = f, t

print('=' * 75)
print(f'  10-FOLD TWO-STAGE STACKING OOF ROC-AUC: {roc_auc_score(y, oof_stacked):.5f}')
print(f'  🏆 OPTIMAL F1 THRESHOLD:                 t = {best_t:.4f}')
print(f'  🏆 STACKED OOF F1-SCORE:                 F1 = {best_f1:.5f}')
print('=' * 75)

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
binary_preds = (test_stacked >= best_t).astype(int)
sub = pd.DataFrame({'id': test_ids.values, 'TARGET': binary_preds})
sub.to_csv(os.path.join(OUT_DIR, 'submission.csv'), index=False)
print(f'Saved submission.csv with {int(binary_preds.sum()):,} predicted positives ({binary_preds.mean():.2%}).')
